<a href="https://colab.research.google.com/github/frasercrichton/ai-dde-hackthon/blob/feature%2Fleiden-guidelines-doc/team-red/notebooks/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI DDE Hackathon

Update the cell below with your Huggingface token (see: https://huggingface.co/docs/hub/en/security-tokens) and ensure you have permssion to use the LLama 3 Model (https://huggingface.co/meta-llama/Llama-3.1-8B).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git triton


from google.colab import userdata
userdata.get('GITHUB_TOKEN')

!git clone https://{GITHUB_TOKEN}@github.com/frasercrichton/ai-dde-hackthon.git
%cd ai-dde-hackthon
! git checkout feature/leiden-guidelines-doc
%cd team-red
! ls

# Install Poetry
# !curl -sSL https://install.python-poetry.org | python3 -

# # Add Poetry to the PATH
# import os
# os.environ["PATH"] += ":/root/.local/bin"

# !poetry init -n
# !poetry install
# ! poetry add git+https://github.com/huggingface/transformers.git


In [ ]:
import sys
import logging
from pathlib import Path

cwd = Path.cwd()
project_root = f'{cwd.parent}'

sys.path.append(project_root)

print('The project directory is:', project_root)
print('The working directory is:', cwd)


for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
#
# self.logger = logging.getLogger(self.__class__.__name__)

logger = logging.getLogger(__name__)
# Install required packages
# !pip install markitdown
# !pip install git+https://github.com/huggingface/transformers.git triton
!pip install langchain_community
import os
# from markitdown import MarkItDown
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch
from src.tokenizer import Tokenizer
from src.chat_message_history_builder import ChatMessageHistoryBuilder
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
from google.colab import userdata

HF_TOKEN = userdata.get('HUGGING_FACE_HUB_TOKEN')


LLM class:

In [ ]:
# from fuzzywuzzy import fuzz
import torch
from src.tokenizer import Tokenizer
from src.chat_message_history_builder import ChatMessageHistoryBuilder
import re
from transformers import AutoModelForCausalLM
#     'meta-llama/Llama-3.1-8B',


torch.cuda.empty_cache()
class LLM:

    def __init__(self, model_name: str, token: str):
        self.model_name = model_name

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=HF_TOKEN,
            device_map="auto" if device == "cuda" else None,
            torch_dtype=torch.float16,
            max_memory={0: "38GiB"}  # if 40GB available, leave headroom
        ).to(device)
        self.model.eval()
        self.tokenizer = Tokenizer(self.model_name, token=HF_TOKEN)

    def run_prompt(self, prompt):

        inputs = self.tokenizer.tokenize(prompt, max_length=1024)
        input_length = inputs['input_ids'].shape[1]

        if torch.cuda.is_available():
            logger.info('Using GPU.')
            self.model.to('cuda')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}

        # TODO - increase the token limit to allow for more text
        eos_token_id = self.tokenizer.eos_token_id or self.tokenizer.convert_tokens_to_ids('<|end_of_text|>')

        generation_config = {
          'max_new_tokens': 500,
          'eos_token_id': eos_token_id,
          'no_repeat_ngram_size': 3,
          'repetition_penalty': 1.2,
          'pad_token_id': eos_token_id,
          'do_sample': False, # ensures the model stays strictly factual and consistent with the source material plus it always returns the consistency of the reponse.
          # 'num_beams': 3,  # Small beam width for better answers
          # temperature=0.3,
          # do_sample=True,

    #       'temperature': 0.0,
    # 'top_p': 1.0,
    # 'top_k': 50,


        }

        with torch.no_grad():
            outputs = self.model.generate(
              **inputs,
              **generation_config
            )
            new_tokens = outputs[0, input_length:]
            response = self.tokenizer.decode(new_tokens).strip()
            print(f'answer {response}')
            print(f'------')
        return response



llm = LLM('meta-llama/Llama-3.1-8B', token=HF_TOKEN)


In [ ]:
system_prompt = """You are an Lllama LLM expert

"""

chatMessageHistoryBuilder = ChatMessageHistoryBuilder(system_prompt)
question = 'As an LLM you repeat text in your responses and misspell words - what prompt could I use to stop you doing that?'
prompt = chatMessageHistoryBuilder.build_prompt('', question)

response = llm.run_prompt(prompt)
answer = response.split("Answer:")[-1].strip()
print(answer)



In [ ]:
system_prompt = """You are a legal assistant analyzing the Leiden Guidelines on digitally derived evidence.
      Answer the question CONCISELY using ONLY the provided document excerpt.
      You MUST include specific requirements when mentioned.
      Do NOT generate any additional information or options.
      Always respond in one of the following exact formats, based on the document:

        Example A:
        "According to the document, video segments must be translated into a working language of the Court to be admissible."

        Example B:
        "The document does not contain enough information to answer this question."

"""

chatMessageHistoryBuilder = ChatMessageHistoryBuilder(system_prompt)
question = 'what language must videos be in to be admissable?'

context =  "Translation. Pursuant to Regulation 39(1) of the Regulations of the Court, all documents and materials filed with the Registry shall be in a working language of the Court. If segments of the video are not in a working language of the Court, those segments must be translated into a working language of the Court before they can be deemed admissible."
prompt = chatMessageHistoryBuilder.build_prompt(context, question)
response = llm.run_prompt(prompt)
answer = response.split("Answer:")[-1].strip()
print(answer)

second_prompt = chatMessageHistoryBuilder.add_user_message(question)
second_prompt = chatMessageHistoryBuilder.add_assistant_message(answer)

second_prompt = chatMessageHistoryBuilder.build_prompt(context, 'under what Regulation?')

response = llm.run_prompt(second_prompt)
answer = response.split("Answer:")[-1].strip()
print(answer)

In [ ]:
# vecorisation could be improved

class DocumentProcessor:
    def __init__(self):
        """Initialize the document processor with necessary components."""
        # Set up embedding model
        self.tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        self.model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        self.model.eval()
        # Initialize document converter
        self.md = MarkItDown()
        # Set up vector database

    def process_document(self, file_path):
        """Convert document to text and generate embeddings."""
        try:
            pdf_processor = PDFProccessor(file_path)
            # Convert document to text
            conversion_result = self.md.convert(file_path)
            conversion_result_text = self.md.convert(file_path).text_content
            # TODO - teh leiden guidelines contain a section of keywwords for each section - these should be parsed out and each section should be stored seperately
            conversion_result_text = pdf_processor.remove_page_numbers(conversion_result_text)
            print(conversion_result)

            # Create embeddings
            inputs = self.tokenizer(
                conversion_result_text,
                return_tensors='pt',
                truncation=True
            )
            # Use GPU if available
            if torch.cuda.is_available():
                self.model.to('cuda')
                inputs = {k: v.to('cuda') for k, v in inputs.items()}
            # Generate embeddings
            with torch.no_grad():
                outputs = self.model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().tolist()
            # **************
            return {
                'text': conversion_result_text,
                'embeddings': embeddings,
                'metadata': {}
            }
            # Note: we don't seem to get metadata from the docs anyway so better manually adding
            getattr(conversion_result, 'metadata', {})
        except Exception as e:
            logger.error(f'Error processing document {file_path}: {str(e)}')
            raise

# Initialize processor
processor = DocumentProcessor()
# Move to GPU if available
if torch.cuda.is_available():
    processor.model = processor.model.to('cuda')

Question-Answering Function

This cell defines the function that generates answers using LLaMA. You may alter the values if you know what you’re doing :)

In [ ]:
# !pip install gradio
import gradio as gr
def ask_question_llama():
  pass
def create_interface():
    demo = gr.Interface(
        fn=ask_question_llama,
        inputs=[
            gr.Dropdown(
            ["Lawyer", "Open Source Researcher", "Journalist"], label="Role", info="How would you describe your role?"),
             gr.Dropdown(
            ['Videos',
             'Photographs',
             'Aerial and Satellite Images',
             'Intercepts',
             'Call Data Records',
             'Audio Recordings'],
            label="Evidence",
            info="What form of Digitally Derived Evidence are you interested in?"),
            gr.Textbox(
              label='Your Question',
              placeholder='Ask any question about Digitally Derived Evidence...',
              lines=3
          ),

            ],

        outputs=[
            gr.Markdown(
                label='Answer',
            )
        ],

        title='Digitally Derived Evidence',
        description='This AI assistant can answer questions Digitally Derived Evidence and particularly the Leiden Guidelines.'
    )
    return demo
# Launch interface
demo = create_interface()
demo.launch(share=True)